In [1]:
import os
import kagglehub
import scipy.io as sio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy import signal

# DEAP sampling rate and EEG frequency bands relevant for emotion.
# Defined here (not later) so every cell below can use them in any run order.
sampling_rate = 128
bands = {
    'Theta': (4, 8),
    'Alpha': (8, 12),
    'Beta':  (12, 30),
    'Gamma': (30, 45),
}

def extract_band_power(eeg_trial, band, fs=128):
    """Return per-channel band power (variance of bandpass-filtered signal).

    eeg_trial : array (n_channels, n_samples)
    band      : (low_hz, high_hz)
    """
    low, high = band
    b, a = signal.butter(4, [low, high], btype='band', fs=fs)
    filtered = signal.filtfilt(b, a, eeg_trial, axis=1)
    return np.var(filtered, axis=1)

C:\Users\Cyberhell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Download latest version
path = kagglehub.dataset_download("manh123df/deap-dataset")
print("Dataset downloaded to:", path)

# Explore what's in the dataset to find the correct folder
print("\nContents of downloaded dataset:")
for item in os.listdir(path):
    print(f"  - {item}")

# The preprocessed data might be in different possible locations
possible_folders = [
    os.path.join(path, 'data_preprocessed_python'),
    os.path.join(path, 'preprocessed_data'),
    path  # Maybe directly in the main folder
]

eeg_folder = None
for folder in possible_folders:
    if os.path.exists(folder):
        # Check if it contains .dat files
        try:
            files = os.listdir(folder)
            if any(f.endswith('.dat') for f in files):
                eeg_folder = folder
                print(f"\n✅ Found .dat files in: {eeg_folder}")
                break
        except:
            pass

if eeg_folder is None:
    print("\n❌ Could not find .dat files. Searching recursively...")
    for root, dirs, files in os.walk(path):
        if any(f.endswith('.dat') for f in files):
            eeg_folder = root
            print(f"✅ Found .dat files in: {eeg_folder}")
            break

# Now continue with your code
if eeg_folder:
    all_dat_files = [f for f in os.listdir(eeg_folder) if f.endswith('.dat')]
    all_dat_files.sort()
    print(f"\nFound {len(all_dat_files)} .dat files")
    print(f"First 15 files: {all_dat_files[:15]}")
    
    # Load first 15 files
    files_to_load = all_dat_files[:15]
    # ... rest of your loading code
else:
    print("No .dat files found in the dataset!")

Dataset downloaded to: C:\Users\Cyberhell\.cache\kagglehub\datasets\manh123df\deap-dataset\versions\1

Contents of downloaded dataset:
  - deap-dataset

❌ Could not find .dat files. Searching recursively...
✅ Found .dat files in: C:\Users\Cyberhell\.cache\kagglehub\datasets\manh123df\deap-dataset\versions\1\deap-dataset\data_preprocessed_python

Found 32 .dat files
First 15 files: ['s01.dat', 's02.dat', 's03.dat', 's04.dat', 's05.dat', 's06.dat', 's07.dat', 's08.dat', 's09.dat', 's10.dat', 's11.dat', 's12.dat', 's13.dat', 's14.dat', 's15.dat']


In [3]:
# List all contents of the main path
print("\nContents of main path:")
for item in os.listdir(path):
    item_path = os.path.join(path, item)
    print(f"  - {item} (is folder: {os.path.isdir(item_path)})")

    # If it's a folder, look inside
    if os.path.isdir(item_path):
        print(f"    Inside {item}:")
        for subitem in os.listdir(item_path):
            print(f"      - {subitem}")




Contents of main path:
  - deap-dataset (is folder: True)
    Inside deap-dataset:
      - audio_stimuli_MIDI
      - audio_stimuli_MIDI_tempo24
      - data_preprocessed_python
      - EDA_DEAP.ipynb
      - Metadata
      - metadata_xls


In [4]:
# Resolve the .dat folder dynamically so this works on Windows AND Colab.
# `path` was defined in cell 1 by kagglehub.dataset_download(...).
eeg_folder = None
for root, dirs, files in os.walk(path):
    if any(f.endswith('.dat') for f in files):
        eeg_folder = root
        break

if eeg_folder is None:
    raise FileNotFoundError(f"No .dat files found anywhere under {path}")
print(f"Using eeg_folder: {eeg_folder}")
print(f"Folder exists: {os.path.exists(eeg_folder)}")

# Get all .dat files
all_dat_files = [f for f in os.listdir(eeg_folder) if f.endswith('.dat')]
all_dat_files.sort()

print(f"Total .dat files found: {len(all_dat_files)}")

# CHANGE THIS LINE - Load ALL files instead of first 15
files_to_load = all_dat_files  # Changed from all_dat_files[:15]
print(f"Loading {len(files_to_load)} files: {files_to_load[:5]}... (showing first 5)")

# Load the data
all_eeg = []
all_labels = []
loaded_subjects = []

for file in files_to_load:
    file_path = os.path.join(eeg_folder, file)
    data = np.load(file_path, allow_pickle=True)  # Added .item()
    
    all_eeg.append(data['data'])
    all_labels.append(data['labels'])
    loaded_subjects.append(file.replace('.dat', ''))
    
    print(f"✓ Loaded {file} - EEG: {data['data'].shape}, Labels: {data['labels'].shape}")

# Combine all subjects
combined_eeg = np.concatenate(all_eeg, axis=0)
combined_labels = np.concatenate(all_labels, axis=0)

print(f"\n{'='*50}")
print(f"SUCCESS! Final dataset:")
print(f"  Subjects loaded: {len(loaded_subjects)}")  # Should be 32
print(f"  Total trials: {combined_eeg.shape[0]}")     # Should be 1280 (32 × 40)
print(f"  EEG shape: {combined_eeg.shape}")           # (1280, 40, 8064)
print(f"  Labels shape: {combined_labels.shape}")     # (1280, 4)
print(f"{'='*50}")

Folder exists: True
Total .dat files found: 32
Loading 32 files: ['s01.dat', 's02.dat', 's03.dat', 's04.dat', 's05.dat']... (showing first 5)
✓ Loaded s01.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s02.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s03.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s04.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s05.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s06.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s07.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s08.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s09.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s10.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s11.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s12.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s13.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s14.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s15.dat - EEG: (40, 40, 8064), Labels: (40, 4)
✓ Loaded s16.dat -

In [5]:
# This should work now
print(f"✓ Combined EEG shape: {combined_eeg.shape}")
print(f"✓ Combined labels shape: {combined_labels.shape}")
print(f"✓ EEG data type: {combined_eeg.dtype}")
print(f"✓ Label range: {combined_labels.min():.1f} to {combined_labels.max():.1f}")

✓ Combined EEG shape: (1280, 40, 8064)
✓ Combined labels shape: (1280, 4)
✓ EEG data type: float64
✓ Label range: 0.0 to 9.0


In [6]:
# (The old 32-channel pruning + 128-feature extraction lived here.
#  It's removed because nothing downstream uses it: cell 6 selects the
#  14 Emotiv channels directly from combined_eeg, and cell 7 builds the
#  56-feature matrix from there.)
print(f"combined_eeg shape: {combined_eeg.shape}  (trials, channels, samples)")

combined_eeg shape: (1280, 40, 8064)  (trials, channels, samples)


In [7]:
# Prune the Emotive EPOC x Channels 
# Headdset has following channels

emotiv_channels = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

# DEAP's 32 channels in order (based on the standard 10-20 system)
deap_32_channels = [
    'Fp1', 'AF3', 'F7', 'F3', 'FC1', 'FC5', 'T7', 'C3',
    'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz',
    'Fp2', 'AF4', 'F8', 'F4', 'FC2', 'FC6', 'T8', 'C4',
    'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'Fz', 'Cz'
]

#Hardcoding the indices
emotiv_indices = [1, 2, 3, 5, 6, 11, 13, 29, 27, 22, 21, 19, 18, 17]

print(f"Using indices: {emotiv_indices}")
print(f"Channel names: {[deap_32_channels[i] for i in emotiv_indices]}")

# Prune the data
eeg_emotiv = combined_eeg[:, emotiv_indices, :]
print(f"Pruned data shape: {eeg_emotiv.shape}")  # (600, 14, 8064)



Using indices: [1, 2, 3, 5, 6, 11, 13, 29, 27, 22, 21, 19, 18, 17]
Channel names: ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
Pruned data shape: (1280, 14, 8064)


In [8]:
# STEP 2 (REWRITTEN AGAIN): 4s windows with 1s hop (overlapping) + DE + DASM features.
#
# Why these specific changes:
#   - 4s windows have 4x the frequency resolution of 2s windows, which matters
#     a lot for theta (4-8 Hz). 2s windows only fit 8-16 cycles of theta, so
#     band-power estimates are noisy; 4s windows fit 16-32 cycles.
#   - 1s hop gives ~57 windows per trial -> 73,000 training rows (~2x more data)
#     and gives the model overlapping views of every moment.
#   - DASM (Differential Asymmetry) = DE(left) - DE(right) for each L/R Emotiv
#     pair, per band. Frontal alpha asymmetry is THE established neural marker
#     of valence (Davidson 1992, Coan & Allen 2004). Adding 28 DASM features
#     gives the model an explicit representation of left-right hemisphere
#     activity differences without making it learn that combination from raw DE.
#
# Total features per window: 56 DE + 28 DASM = 84.

WINDOW_SECONDS = 4
HOP_SECONDS    = 1
WINDOW_SAMPLES = WINDOW_SECONDS * sampling_rate           # 512
HOP_SAMPLES    = HOP_SECONDS    * sampling_rate           # 128
BASELINE_SAMPLES = 3 * sampling_rate                       # 384 (DEAP pre-stimulus baseline)
N_BANDS = len(bands)                                       # 4
N_CH = eeg_emotiv.shape[1]                                 # 14

# DEAP files are loaded in order s01..s32, 40 trials each.
N_SUBJECTS = 32
TRIALS_PER_SUBJECT = 40

# Drop the 3s pre-stimulus baseline -> 60s of actual stimulus per trial
stimulus = eeg_emotiv[:, :, BASELINE_SAMPLES:]             # (1280, 14, 7680)
n_trials_orig = stimulus.shape[0]
n_samples_per_trial = stimulus.shape[2]

# Number of overlapping windows per trial: floor((N - W)/H) + 1
n_windows_per_trial = (n_samples_per_trial - WINDOW_SAMPLES) // HOP_SAMPLES + 1   # 57
n_total_windows = n_trials_orig * n_windows_per_trial                              # 72960

# Index arrays that map each window back to its parent trial / subject
trial_idx_per_window = np.repeat(np.arange(n_trials_orig), n_windows_per_trial)
subject_id_per_trial = np.arange(n_trials_orig) // TRIALS_PER_SUBJECT
subject_id_per_window = subject_id_per_trial[trial_idx_per_window]

print(f"Stimulus per trial: {n_samples_per_trial} samples ({n_samples_per_trial/sampling_rate:.0f}s)")
print(f"Windows per trial:  {n_windows_per_trial} x {WINDOW_SECONDS}s (hop {HOP_SECONDS}s, overlapping)")
print(f"Total windows:      {n_total_windows}")

# Left-right Emotiv channel pairs (indices into self.channels order).
# AF3-AF4, F7-F8, F3-F4, FC5-FC6, T7-T8, P7-P8, O1-O2  ->  7 pairs.
LEFT_IDX  = np.array([0, 1, 2, 3, 4, 5, 6])
RIGHT_IDX = np.array([13, 12, 11, 10, 9, 8, 7])
N_PAIRS   = len(LEFT_IDX)                                 # 7
N_FEATURES = N_BANDS * (N_CH + N_PAIRS)                   # 4 * (14 + 7) = 84

# Pre-build the bandpass filters once
band_filters = {name: signal.butter(4, [lo, hi], btype='band', fs=sampling_rate)
                for name, (lo, hi) in bands.items()}

def compute_features(window):
    """window: (n_channels, n_samples) -> 84 features.
    Layout (band-major to match cell 12's feature_names):
        [DE_Theta x14 | DE_Alpha x14 | DE_Beta x14 | DE_Gamma x14 |
         DASM_Theta x7 | DASM_Alpha x7 | DASM_Beta x7 | DASM_Gamma x7]
    """
    de   = np.empty(N_CH * N_BANDS, dtype=np.float64)
    dasm = np.empty(N_PAIRS * N_BANDS, dtype=np.float64)
    for bi, (b, a) in enumerate(band_filters.values()):
        filtered = signal.filtfilt(b, a, window, axis=1)
        var = np.var(filtered, axis=1) + 1e-12
        de_band = 0.5 * np.log(2 * np.pi * np.e * var)
        de[bi * N_CH:(bi + 1) * N_CH] = de_band
        dasm[bi * N_PAIRS:(bi + 1) * N_PAIRS] = de_band[LEFT_IDX] - de_band[RIGHT_IDX]
    return np.concatenate([de, dasm])

print(f"\nComputing DE + DASM features for all windows ({N_FEATURES} features each)...")
features_emotiv = np.zeros((n_total_windows, N_FEATURES), dtype=np.float64)
write_idx = 0
for trial_idx in range(n_trials_orig):
    trial = stimulus[trial_idx]                            # (14, 7680)
    for w in range(n_windows_per_trial):
        s = w * HOP_SAMPLES
        features_emotiv[write_idx] = compute_features(trial[:, s:s + WINDOW_SAMPLES])
        write_idx += 1
    if (trial_idx + 1) % 100 == 0:
        print(f"  {trial_idx + 1}/{n_trials_orig} trials processed ({write_idx}/{n_total_windows} windows)")

print(f"\nFeature matrix shape: {features_emotiv.shape}  (windows, 56 DE + 28 DASM)")

Stimulus per trial: 7680 samples (60s)
Windows per trial:  30 x 2s
Total windows:      38400  (shape (38400, 14, 256))

Computing DE features for all windows...
  5000/38400
  10000/38400
  15000/38400
  20000/38400
  25000/38400
  30000/38400
  35000/38400

DE feature matrix shape: (38400, 56)  (windows, 14 ch x 4 bands)


In [9]:
# STEP 3: Normalize features
#
# Two-stage normalization:
#   (a) Per-subject z-score: each subject's feature distribution is centered
#       on its own mean/std. This removes between-subject offsets (skull
#       thickness, electrode contact, individual band baselines) which would
#       otherwise dominate the model's "this is positive valence" signal.
#   (b) A tiny global StandardScaler on top. After (a) the data is already
#       roughly N(0,1), so the global scaler is near-identity, but it gives
#       us a saved object the realtime recognizer can use as a sanity layer.
#
# Per-subject means/stds are also kept around so the realtime recognizer's
# self-calibration step (see cell 16) can mimic this transformation against
# whoever is wearing the headset.
from sklearn.preprocessing import StandardScaler

features_zscored = np.zeros_like(features_emotiv)
subject_feature_stats = {}                 # id -> (mean[56], std[56])
for s in np.unique(subject_id_per_window):
    mask = subject_id_per_window == s
    sub_mean = features_emotiv[mask].mean(axis=0)
    sub_std = features_emotiv[mask].std(axis=0) + 1e-8
    features_zscored[mask] = (features_emotiv[mask] - sub_mean) / sub_std
    subject_feature_stats[int(s)] = (sub_mean, sub_std)

scaler = StandardScaler()
features_normalized = scaler.fit_transform(features_zscored)

print(f"After per-subject z-score: mean={features_zscored.mean():.4f}, std={features_zscored.std():.4f}")
print(f"After global scaler:       mean={features_normalized.mean():.4f}, std={features_normalized.std():.4f}")

Normalized features - mean: 0.000000, std: 1.000000


In [10]:
# STEP 4: Binarize VALENCE PER SUBJECT against that subject's own median.
#
# Why not a fixed threshold of 5?  Subject 12 rated only 5% of trials > 5,
# subject 3 rated 52.5% > 5. A global cutoff puts subject 12's "actually
# slightly happy" trials in the negative class and subject 3's "actually
# neutral" trials in the positive class - the model is being asked to
# memorise contradictions. Per-subject median binarization makes every
# subject contribute ~50% positive / ~50% negative, removes the global
# imbalance entirely, and learns the within-subject "above your average
# mood vs below" signal which is what valence really is.
valence_raw = combined_labels[:, 0]
subject_medians = np.array([
    np.median(valence_raw[subject_id_per_trial == s]) for s in range(N_SUBJECTS)
])
trial_thresholds = subject_medians[subject_id_per_trial]                   # (1280,)
valence_trial_binary = (valence_raw > trial_thresholds).astype(int)        # (1280,)
valence_binary = valence_trial_binary[trial_idx_per_window]                # (38400,)

n_win = len(valence_binary)
print(f"Per-subject median range: [{subject_medians.min():.1f} .. {subject_medians.max():.1f}]")
print("Window-level label distribution after per-subject median binarization:")
print(f"  Positive (above own median): {valence_binary.sum():>5d}  ({valence_binary.mean()*100:.1f}%)")
print(f"  Negative (at/below median):  {n_win - valence_binary.sum():>5d}  ({(1 - valence_binary.mean())*100:.1f}%)")

Window-level label distribution:
  Valence - Positive (1):  8070  (21.0%)
  Valence - Negative (0): 30330  (79.0%)


In [11]:
X_cnn = features_normalized.reshape(features_normalized.shape[0], features_normalized.shape[1], 1)
print(f"CNN input shape: {X_cnn.shape}")  # (1280, 56, 1)

CNN input shape: (38400, 56, 1)


In [12]:
# STEP 6: Train/Test split with stratification
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_val, y_test_val = train_test_split(
    X_cnn, valence_binary, test_size=0.2, random_state=42, stratify=valence_binary
)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"Valence train positive: {np.sum(y_train_val)}/{len(y_train_val)} ({np.sum(y_train_val)/len(y_train_val)*100:.1f}%)")
print(f"Valence test positive: {np.sum(y_test_val)}/{len(y_test_val)} ({np.sum(y_test_val)/len(y_test_val)*100:.1f}%)")


Training samples: 30720, Test samples: 7680
Valence train positive: 6456/30720 (21.0%)
Valence test positive: 1614/7680 (21.0%)


In [13]:
# Build a human-readable name for each of the 84 features.
# Order MUST match the extraction in cell 7:
#   [DE_band1 x 14ch | DE_band2 x 14ch | ... | DASM_band1 x 7pairs | ...]
de_names   = [f"DE_{band}_{ch}"
              for band in bands.keys()
              for ch in emotiv_channels]
dasm_pairs = [(emotiv_channels[l], emotiv_channels[r])
              for l, r in zip(LEFT_IDX, RIGHT_IDX)]
dasm_names = [f"DASM_{band}_{lname}-{rname}"
              for band in bands.keys()
              for (lname, rname) in dasm_pairs]
feature_names = de_names + dasm_names

print(f"Total features: {len(feature_names)}  (56 DE + 28 DASM)")
print("First 4 DE features:  ", feature_names[:4])
print("First 4 DASM features:", feature_names[56:60])

Total features: 56
First 8 feature names: ['AF3_Theta', 'F7_Theta', 'F3_Theta', 'FC5_Theta', 'T7_Theta', 'P7_Theta', 'O1_Theta', 'O2_Theta']


In [ ]:
# STEP 7 (replaces the old CNN cell): train XGBoost on the 84 DE+DASM features.
#
# XGBoost was added as a sanity-check baseline and ended up beating the CNN
# by ~10pp on this feature set. It is now the only valence model in this
# notebook. Cells 15-17 read the predictions exposed at the end of this cell
# as `y_pred_val` and `y_pred_val_prob`.

try:
    import xgboost as xgb
except ImportError:
    print("Installing xgboost...")
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Cell 11's X_train / X_test are (N, 84, 1) for the CNN; flatten the trailing dim
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0],  -1)

# Hold out a val set from training rows (same ratio as the CNN's internal
# val split) so XGBoost has its own early-stopping signal that doesn't
# touch the test set.
from sklearn.model_selection import train_test_split as _tts_xgb
X_xtr, X_xva, y_xtr, y_xva = _tts_xgb(
    X_train_flat, y_train_val,
    test_size=0.2, random_state=42, stratify=y_train_val
)

xgb_model = xgb.XGBClassifier(
    n_estimators=3000,           # bumped from 1000 - last run never converged
    max_depth=6,
    learning_rate=0.03,           # lowered from 0.05 to make use of the extra trees
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',           # fast histogram-based; uses GPU if available
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=50,     # bumped from 30 to give the slower LR room
)

print("\n" + "="*60)
print("Training XGBoost on 84 DE+DASM features")
print("="*60)
xgb_model.fit(X_xtr, y_xtr, eval_set=[(X_xva, y_xva)], verbose=50)
print(f"Best iteration: {xgb_model.best_iteration} of {xgb_model.n_estimators}")

# Evaluate on the held-out TEST set
xgb_probs = xgb_model.predict_proba(X_test_flat)[:, 1]
xgb_preds = (xgb_probs > 0.5).astype(int)

print("\n" + "="*60)
print("XGBoost Test Results (threshold 0.5)")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test_val, xgb_preds):.4f}")
print(f"Precision: {precision_score(y_test_val, xgb_preds, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test_val, xgb_preds, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test_val, xgb_preds, zero_division=0):.4f}")

# Expose XGBoost predictions as the canonical y_pred_val / y_pred_val_prob
# so cell 15 (full evaluation) consumes them without any further changes.
y_pred_val      = xgb_preds
y_pred_val_prob = xgb_probs

# ---------------- Which features actually matter ----------------
top_n = 12
imp = xgb_model.feature_importances_
top_idx = np.argsort(imp)[::-1][:top_n]
print(f"\nTop {top_n} most important features:")
for rank, idx in enumerate(top_idx, 1):
    bar = '#' * int(imp[idx] / imp[top_idx[0]] * 30)
    print(f"  {rank:>2}. {feature_names[idx]:<28s} {imp[idx]:.4f}  {bar}")


In [ ]:
# STEP 9: Full evaluation - confusion matrix, classification report,
# and training curves for the VALENCE model.

from sklearn.metrics import confusion_matrix, classification_report

def evaluate_model(name, y_true, y_pred, y_prob, class_labels):
    print("\n" + "="*60)
    print(f"{name} - Full Evaluation")
    print("="*60)
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix:")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}  TP={tp}")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels, zero_division=0))
    return cm

cm_val = evaluate_model(
    "VALENCE",
    y_test_val, y_pred_val, y_pred_val_prob,
    class_labels=['Negative (0)', 'Positive (1)']
)

# Visualise confusion matrix + XGBoost training (logloss vs boosting round).
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
axes[0].set_title('Valence Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# XGBoost stores its eval history under .evals_result()
evals = xgb_model.evals_result() if hasattr(xgb_model, 'evals_result') else None
if evals and 'validation_0' in evals:
    metric = next(iter(evals['validation_0']))   # usually 'logloss'
    axes[1].plot(evals['validation_0'][metric], label=f'val {metric}')
    if xgb_model.best_iteration is not None:
        axes[1].axvline(xgb_model.best_iteration, color='r', linestyle='--',
                        alpha=0.6, label=f'best iter ({xgb_model.best_iteration})')
    axes[1].set_title('XGBoost Validation Loss vs Boosting Round')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel(metric)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
else:
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, 'No XGBoost eval history available',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()


In [ ]:
# STEP 10: Save the trained valence model and scaler for real-time inference.
# Constants (sampling rate, channel order, bands, window length) are NOT saved -
# they are hardcoded in the recognizer in the next cell since they never change.
import joblib

# Save under /content on Colab, under the user's Windows folder otherwise.
ARTIFACT_DIR = '/content/eeg_emotion' if os.path.isdir('/content') else r'C:\Users\Cyberhell\eeg_emotion'
MODELS_DIR = os.path.join(ARTIFACT_DIR, 'models')
ARTIFACTS_DIR = os.path.join(ARTIFACT_DIR, 'artifacts')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

valence_xgb_path = os.path.join(MODELS_DIR, 'valence_xgb.joblib')
scaler_path      = os.path.join(ARTIFACTS_DIR, 'scaler.joblib')

joblib.dump(xgb_model, valence_xgb_path)
joblib.dump(scaler, scaler_path)

print("Saved artifacts:")
print(f"  Valence XGBoost -> {valence_xgb_path}")
print(f"  Scaler          -> {scaler_path}")


In [ ]:
# STEP 11: Real-time inference scaffold for Emotiv EPOC X
#
# This cell defines a reusable real-time inference class that:
#   1. Loads the saved valence XGBoost model + scaler
#   2. Pulls live EEG samples (BrainFlow or a SIMULATED board for testing)
#   3. Maintains a sliding window of 4 s @ 128 Hz (512 samples)
#   4. Expects the 14 Emotiv channels in the SAME order used in training
#   5. Runs the same DE + DASM feature extraction (84 features)
#   6. Per-user baseline z-score (mirrors per-subject z-score in cell 8)
#
# Per-user fine-tuning is intentionally NOT included here. With XGBoost
# you cannot fine-tune the way you can with a neural network's final
# layer. If you ever need wearer-specific adaptation, the standard moves
# are: (a) refit a small calibration model (e.g. logistic regression) on
# top of XGBoost's predicted probability using a few labeled wearer
# windows, or (b) re-train XGBoost from scratch with the wearer's data
# folded into the training set.
#
# IMPORTANT NOTES ABOUT EMOTIV + BRAINFLOW
# - Emotiv EPOC X is NOT natively supported by BrainFlow. Two practical paths:
#     (a) Use the official Emotiv Cortex SDK (websocket) and feed samples into
#         RealTimeEmotionRecognizer.push_sample(...) yourself.
#     (b) Have EmotivPRO/Cortex push the EEG stream over LSL and read it via
#         BrainFlow's STREAMING_BOARD - the cleanest BrainFlow-compatible route.
# - For development without hardware, use BoardIds.SYNTHETIC_BOARD. The code below
#   will tile its 8 fake EEG channels to 14 just so plumbing runs (predictions on
#   random data are meaningless - this is only to verify the pipeline).

import os
import time
import threading
import collections
import numpy as np
import joblib
from scipy import signal

# Save under /content on Colab, under the user's Windows folder otherwise.
ARTIFACT_DIR = '/content/eeg_emotion' if os.path.isdir('/content') else r'C:\Users\Cyberhell\eeg_emotion'
MODELS_DIR = os.path.join(ARTIFACT_DIR, 'models')
ARTIFACTS_DIR = os.path.join(ARTIFACT_DIR, 'artifacts')


class RealTimeEmotionRecognizer:
    """Sliding-window real-time emotion recognizer matching the DEAP training pipeline."""

    # These are the same constants used during training - intentionally hardcoded
    # so the recognizer is self-contained and doesn't depend on any sidecar file.
    # NOTE: window is now 4s (matches the 4s/1s overlapping pipeline in cell 7).
    FS = 128
    WINDOW_SECONDS = 4
    CHANNELS = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1',
                'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
    BANDS = {
        'Theta': (4, 8),
        'Alpha': (8, 12),
        'Beta':  (12, 30),
        'Gamma': (30, 45),
    }
    # Left-right pairs for DASM, indices into CHANNELS (must match cell 7)
    LEFT_IDX  = [0, 1, 2, 3, 4, 5, 6]
    RIGHT_IDX = [13, 12, 11, 10, 9, 8, 7]

    def __init__(self,
                 valence_xgb_path=os.path.join(MODELS_DIR, 'valence_xgb.joblib'),
                 scaler_path=os.path.join(ARTIFACTS_DIR, 'scaler.joblib')):
        self.valence_model = joblib.load(valence_xgb_path)
        self.scaler = joblib.load(scaler_path)

        self.fs = self.FS
        self.channels = list(self.CHANNELS)
        self.bands = dict(self.BANDS)
        self.window_seconds = self.WINDOW_SECONDS
        self.window_size = self.window_seconds * self.fs   # 8064
        self.n_channels = len(self.channels)               # 14

        # Sliding buffer per channel
        self._buffers = [collections.deque(maxlen=self.window_size)
                         for _ in range(self.n_channels)]
        self._lock = threading.Lock()

        # Per-user calibration state (mirrors per-subject z-scoring in cell 8).
        # Collect ~30s of feature vectors at startup to estimate this wearer's
        # baseline, then z-score all subsequent predictions against it.
        # 30s / WINDOW_SECONDS gives the target count.
        self._calib_target_windows = max(8, int(30 / self.window_seconds))
        self._calib_features = []
        self._calib_mean = None
        self._calib_std = None

        # Pre-build bandpass filters once
        self._band_filters = {name: signal.butter(4, [lo, hi], btype='band', fs=self.fs)
                              for name, (lo, hi) in self.bands.items()}

    def is_calibrated(self):
        return self._calib_mean is not None

    def calibration_progress(self):
        if self.is_calibrated():
            return 1.0
        return len(self._calib_features) / self._calib_target_windows

    def reset_calibration(self):
        """Throw away the current per-user baseline and re-calibrate from scratch."""
        self._calib_features = []
        self._calib_mean = None
        self._calib_std = None

    def _extract_features(self, window):
        """window: (n_channels, n_samples) -> 84 features (56 DE + 28 DASM).
        Layout matches training cell 7."""
        n_ch = self.n_channels
        n_pairs = len(self.LEFT_IDX)
        de   = np.empty(n_ch * 4, dtype=np.float64)
        dasm = np.empty(n_pairs * 4, dtype=np.float64)
        for bi, (b, a) in enumerate(self._band_filters.values()):
            filtered = signal.filtfilt(b, a, window, axis=1)
            var = np.var(filtered, axis=1) + 1e-12
            de_band = 0.5 * np.log(2 * np.pi * np.e * var)
            de[bi * n_ch:(bi + 1) * n_ch] = de_band
            dasm[bi * n_pairs:(bi + 1) * n_pairs] = (
                de_band[self.LEFT_IDX] - de_band[self.RIGHT_IDX]
            )
        return np.concatenate([de, dasm])

    def _get_window(self):
        with self._lock:
            return np.array([list(b) for b in self._buffers])  # (14, window_size)

    def _normalize(self, feats):
        """Apply per-user z-score then the saved global StandardScaler."""
        feats_calib = (feats - self._calib_mean) / self._calib_std
        return self.scaler.transform(feats_calib.reshape(1, -1))

    def push_sample(self, sample):
        """Push ONE sample. `sample` must be a length-14 array in self.channels order."""
        if len(sample) != self.n_channels:
            raise ValueError(f"Expected {self.n_channels} channels, got {len(sample)}")
        with self._lock:
            for ch_idx, val in enumerate(sample):
                self._buffers[ch_idx].append(float(val))

    def push_chunk(self, chunk):
        """Push a chunk of shape (n_channels, n_samples) in self.channels order."""
        chunk = np.asarray(chunk)
        if chunk.shape[0] != self.n_channels:
            raise ValueError(f"Expected {self.n_channels} channels, got shape {chunk.shape}")
        with self._lock:
            for ch_idx in range(self.n_channels):
                self._buffers[ch_idx].extend(chunk[ch_idx].tolist())

    def _ready(self):
        return all(len(b) == self.window_size for b in self._buffers)

    def predict(self):
        """Run one inference step on the current window.
        Returns None until the buffer is full, then a dict with a 'status' key:
          - {'status': 'calibrating', 'progress': 0..1}   while baseline is forming
          - {'status': 'predicting',  'valence_prob': ..., 'valence': 'positive'/'negative'}
        """
        if not self._ready():
            return None
        feats = self._extract_features(self._get_window())

        # Phase-1 calibration: collect baseline, no prediction yet
        if not self.is_calibrated():
            self._calib_features.append(feats.copy())
            if len(self._calib_features) >= self._calib_target_windows:
                arr = np.array(self._calib_features)
                self._calib_mean = arr.mean(axis=0)
                self._calib_std = arr.std(axis=0) + 1e-8
            return {
                'status': 'calibrating',
                'progress': self.calibration_progress(),
                'timestamp': time.time(),
            }

        feats_norm = self._normalize(feats)
        v_prob = float(self.valence_model.predict_proba(feats_norm)[0, 1])
        return {
            'status': 'predicting',
            'valence_prob': v_prob,
            'valence': 'positive' if v_prob > 0.5 else 'negative',
            'timestamp': time.time(),
        }



# --------------------------------------------------------------------------
# Backend A: BrainFlow streaming loop (works with SYNTHETIC_BOARD for testing,
# and with STREAMING_BOARD when EmotivPRO/Cortex is pushing over LSL).
# --------------------------------------------------------------------------
def run_brainflow_stream(predict_every_seconds=2.0,
                         total_seconds=120,
                         board_id=None,
                         serial_port='',
                         ip_port=0,
                         ip_address=''):
    """
    Run a real-time loop with BrainFlow.

    For EMOTIV EPOC X via LSL (recommended):
        from brainflow.board_shim import BoardIds
        run_brainflow_stream(board_id=BoardIds.STREAMING_BOARD,
                             ip_address='127.0.0.1', ip_port=6677)

    For development without hardware:
        from brainflow.board_shim import BoardIds
        run_brainflow_stream(board_id=BoardIds.SYNTHETIC_BOARD)
    """
    from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds

    if board_id is None:
        board_id = BoardIds.SYNTHETIC_BOARD

    params = BrainFlowInputParams()
    params.serial_port = serial_port
    params.ip_port = ip_port
    params.ip_address = ip_address

    board = BoardShim(board_id, params)
    board_fs = BoardShim.get_sampling_rate(board_id)
    eeg_channel_indices = BoardShim.get_eeg_channels(board_id)

    print(f"BrainFlow board fs = {board_fs} Hz, EEG channel rows = {eeg_channel_indices}")
    print(f"Model expects fs = 128 Hz on 14 channels in order: {recognizer.channels}")
    if board_fs != 128:
        print("WARNING: board sampling rate != 128 Hz. You should resample the chunk "
              "before calling recognizer.push_chunk(...) for accurate inference.")

    board.prepare_session()
    board.start_stream()
    try:
        end_time = time.time() + total_seconds
        last_predict = 0.0
        while time.time() < end_time:
            time.sleep(0.1)
            data = board.get_board_data()  # (n_rows, n_samples)
            if data.size == 0:
                continue

            # Pick EEG rows. If the board provides >=14 EEG channels, take the first 14.
            # For real EMOTIV data via LSL you must remap to the order recognizer.channels.
            eeg = data[eeg_channel_indices, :]
            if eeg.shape[0] < recognizer.n_channels:
                # SYNTHETIC_BOARD has 8 EEG channels - tile to 14 just so plumbing runs.
                reps = int(np.ceil(recognizer.n_channels / eeg.shape[0]))
                eeg = np.tile(eeg, (reps, 1))[:recognizer.n_channels, :]
            else:
                eeg = eeg[:recognizer.n_channels, :]

            recognizer.push_chunk(eeg)

            now = time.time()
            if now - last_predict >= predict_every_seconds:
                pred = recognizer.predict()
                if pred is None:
                    filled = len(recognizer._buffers[0])
                    print(f"buffering... {filled}/{recognizer.window_size} samples")
                elif pred['status'] == 'calibrating':
                    print(f"[{time.strftime('%H:%M:%S')}] "
                          f"calibrating baseline... {pred['progress']*100:.0f}%")
                else:
                    print(f"[{time.strftime('%H:%M:%S')}] "
                          f"valence={pred['valence']:>8s} ({pred['valence_prob']:.2f})")
                last_predict = now
    finally:
        board.stop_stream()
        board.release_session()


# --------------------------------------------------------------------------
# Backend B: Emotiv Cortex SDK (websocket).
# Sketch only - requires an Emotiv developer account, an app client_id/secret,
# and the `cortex` python wrapper (or write your own websocket client).
# Fill in credentials and the channel-name -> index mapping for your headset.
# --------------------------------------------------------------------------
EMOTIV_CORTEX_SKETCH = r"""
# pip install websocket-client
import json, ssl, websocket

CLIENT_ID = '...'
CLIENT_SECRET = '...'
HEADSET_ID = 'EPOCX-XXXXXXXX'

ws = websocket.create_connection('wss://localhost:6868', sslopt={'cert_reqs': ssl.CERT_NONE})

def rpc(method, params):
    ws.send(json.dumps({'jsonrpc':'2.0','id':1,'method':method,'params':params}))
    return json.loads(ws.recv())

rpc('requestAccess', {'clientId':CLIENT_ID,'clientSecret':CLIENT_SECRET})
auth = rpc('authorize', {'clientId':CLIENT_ID,'clientSecret':CLIENT_SECRET})['result']['cortexToken']
rpc('controlDevice', {'command':'connect','headset':HEADSET_ID})
session = rpc('createSession', {'cortexToken':auth,'headset':HEADSET_ID,'status':'active'})['result']['id']
rpc('subscribe', {'cortexToken':auth,'session':session,'streams':['eeg']})

# First 'eeg' message contains the channel labels in 'cols'. Map them to recognizer.channels:
labels_msg = json.loads(ws.recv())
cols = labels_msg['eeg']['cols']  # e.g. ['COUNTER','INTERPOLATED','AF3','F7',...,'MARKERS']
remap = [cols.index(name) for name in recognizer.channels]

while True:
    msg = json.loads(ws.recv())
    if 'eeg' not in msg:
        continue
    row = msg['eeg']                    # one sample, length == len(cols)
    sample = [row[i] for i in remap]    # length 14, same order as training
    recognizer.push_sample(sample)
    pred = recognizer.predict()
    if pred:
        print(pred)
"""

# --------------------------------------------------------------------------
# Build the recognizer once. Uncomment one of the backends below to actually
# stream. They are commented out so the notebook does not try to open a
# headset session every time the cell is run.
# --------------------------------------------------------------------------
recognizer = RealTimeEmotionRecognizer()
print("Real-time recognizer ready.")
print(f"  Model:    XGBoost ({recognizer.valence_model.n_estimators} trees)")
print(f"  Window:   {recognizer.window_seconds}s @ {recognizer.fs}Hz "
      f"({recognizer.window_size} samples)")
print(f"  Channels: {recognizer.channels}")

# --- Test plumbing with synthetic data (no headset required) ---
# from brainflow.board_shim import BoardIds
# run_brainflow_stream(board_id=BoardIds.SYNTHETIC_BOARD, total_seconds=90)

# --- Real EMOTIV EPOC X via EmotivPRO -> LSL -> BrainFlow STREAMING_BOARD ---
# from brainflow.board_shim import BoardIds
# run_brainflow_stream(board_id=BoardIds.STREAMING_BOARD,
#                      ip_address='127.0.0.1', ip_port=6677,
#                      total_seconds=300)

# --- Or use the Emotiv Cortex sketch in EMOTIV_CORTEX_SKETCH (string above) ---
print("\nTo go live, uncomment one of the run_brainflow_stream(...) calls above,")
print("or implement EMOTIV_CORTEX_SKETCH with your Cortex credentials.")
